This notebook largely follows the approach of ["An NLP-based approach to assessing a company's maturity level in the digital era" (2024) by Romano, Sperlì, and Vignali](https://www.sciencedirect.com/science/article/pii/S0957417424011588)

In [ ]:
from collections import Counter
import re
from sklearn.feature_extraction.text import CountVectorizer
import string

import nltk
nltk.download()
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import PunktSentenceTokenizer, RegexpTokenizer

import os
import sys

# Comment out all but the appropriate option
where_running = "Google Colab"
# where_running = "Local Machine"

if where_running == "Local Machine":
  module_path = os.path.abspath(os.path.join('..'))
  if not module_path in sys.path:
      sys.path.insert(0, module_path)
elif where_running == "Google Colab":
  dirpath = '/content/digi-inno-road-prod'
  if not os.path.isdir(dirpath):
    # TODO git pull
    !git clone https://github.com/roughhawkbit/digi-inno-road-prod.git
  sys.path.insert(0,dirpath)

from innoprod.sheet_tools import get_sheet_dfs
from innoprod.wrangling.msyh_data_sharing import wrangle_roadmaps
from innoprod.wrangling.wrangling_tools import is_non_empty

In [ ]:
data = get_sheet_dfs()
roadmaps_df = wrangle_roadmaps(data['Roadmaps'])

In [ ]:
cols = [
    'Summary review of Edge Digital diagnostic report & current state and key improvement areas',
    'What are the internal barriers to growth? How do you intend to finance future growth? Are there sufficient leadership and management skills in the business to achieve your growth? What opportunities do you have to expand into new markets?',
    'Level of current Strategic Digital Skills/knowledge in the business',
    'Level of current Technical Digital Skills/knowledge in the business',
    'Whether the business is already investing/adopting/utilising Industry 4.0 Technologies, with examples',
    'Summary of the identified problems, including Gap Analysis',
    'Key potential industry 4.0 solutions to address the identified problems/gaps',
    'Recommended Action Plan to utilise Industry 4.0 Technology',
    'Overview of qualitative benefits of recommended Action Plan (positive/negative)',
    'Skills and other requirements that will be needed to successfully implement the recommended Action Plan',
    'What has been your overall opinion of the support you have received in this programme? (Add comments)'
]

In [ ]:
responses_df = roadmaps_df[['Client ID'] + cols].melt(id_vars=['Client ID'], value_vars=cols, var_name='Question', value_name='Response')
responses_df = responses_df[is_non_empty(responses_df['Response'])]

responses_df

In [ ]:
tokenizer = PunktSentenceTokenizer()

sentences_df = responses_df.copy()
sentences_df['Sentences'] = sentences_df.apply(lambda row: tokenizer.tokenize(row['Response']), axis=1)
sentences_df = sentences_df[['Client ID', 'Question', 'Sentences']].explode('Sentences').reset_index().rename(columns={'index' : 'Sentence Number', 'Sentences' : 'Sentence'})
sentences_df['Sentence Number'] = sentences_df.groupby('Sentence Number').cumcount() + 1
sentences_df

# Preprocessing

In [ ]:
acronyms_abbreviations = {
    'I4.0': 'industry 4.0',
    'tech': 'technology',
}

# This is an important step: phrases here will be treated as single tokens, so we can ensure that important concepts are not split up during tokenization.
# This is especially important for multi-word technical terms, which would otherwise be split into their individual words and lose their meaning.
phrases = [
    "business returns",
    "digital roadmap",
    "digital strategy",
    "industry 4.0",
    "strategic skills",
    "technical skills",
]

In [ ]:
stops = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

regexpr = "|".join(phrases) + "|[\\w']+"
tokenizer = RegexpTokenizer(regexpr)

def preprocess_text(text):
    cleaned_text = text.replace("[REDACTED]", "")
    for short_form, full_form in acronyms_abbreviations.items():
        matches = re.findall(f'({short_form})[\\s|{re.escape(string.punctuation)}]', cleaned_text)
        for match in matches:
            cleaned_text = re.sub(re.escape(match), full_form, cleaned_text, flags=re.IGNORECASE)
    # Skipped removing punctuation as tokenization already does this.
    cleaned_text = cleaned_text.lower()
    # TO DO: explore switching the order of the lemmatization and stop word removal steps:
    # e.g., "has" is a stop word, but post-lemmatization it becomes "ha", which is not a stop word, so it is retained in the final output.
    tokenized_answer = tokenizer.tokenize(cleaned_text)
    lemmatized_answer = [lemmatizer.lemmatize(word) for word in tokenized_answer]
    final_answer = [lemma for lemma in lemmatized_answer if lemma not in stops]
    return final_answer

In [ ]:
'has' in stops

In [ ]:
responses_df['Cleaned Response'] = responses_df.apply(lambda row: preprocess_text(row['Response']), axis=1)

In [ ]:
all_words = responses_df['Cleaned Response'].explode()

In [ ]:
vectorizer = CountVectorizer()
transform_array = vectorizer.fit_transform()

# Expert supervision

In [ ]:
concepts = {}

def add_word(word, concept):
    if concept not in concepts:
        concepts[concept] = set()
    concepts[concept].add(word)

In [ ]:
counted_words = Counter(all_words).most_common()

In [ ]:
counted_words[0:20]


In [ ]:
[(word, freq) for word, freq in counted_words if word in phrases]